In [ ]:
import Omniwheel_Protocol
import serial
import time

Arduino_ID = 0x20
REQUEST_Ultra_Sonic_SENSOR = 0xA1
ANSWER_Ultra_Sonic_SENSOR = 0xB1
MID_Ultra_Sonic_SENSOR = 0x80
data_Ultrasonic_Sensors = 0
send_packet = Omniwheel_Protocol.Packet()
recv_packet = Omniwheel_Protocol.Packet()
recv_packet.clearPacket()
send_packet.clearPacket()
recv_list = []
recv_parsing_packet = []
send_flag = False
Serial_Arduino = serial.Serial(port="/dev/ttyUSB0", baudrate=9600, timeout=.1)

time.sleep(1)
print("connect complete")
def Packet_send(_id, _cmd, _mid, _data=None):
    global send_flag
if(send_flag == False):
    send_packet.clearPacket()
    send_packet.setID(_id)
    send_packet.setCMD(_cmd)
    send_packet.clearPayload()
    send_packet.addPayload(_mid, _data)
    send_packet.calcLRC_Lower()
    send_list = send_packet.packetToList()
    Serial_Arduino.write(send_list)
    send_flag = True

def Packet_receive(ser):
    global send_flag

    if(send_flag == True):
        while ser.inWaiting() > 0:
            Arduino_Data = ser.read(1)
            if(len(Arduino_Data) > 0):
                recv_list.append(ord(Arduino_Data))
                if(ord(Arduino_Data) == 0x03):
                    if(recv_packet.parsingList(recv_list)):
                        recv_parsing_packet.append(recv_packet)
                        recv_list.clear()
                        send_flag = False
                        break

def Received_packet():
    result = recv_parsing_packet[0]
    del recv_parsing_packet[0]
    return result

def Ultrasonic_Data(packet):


    global data_Ultrasonic_Sensors
    packet_id = packet.getID()
    packet_cmd = packet.getCMD()

    if(packet_id == Arduino_ID):
        if(packet_cmd == ANSWER_Ultra_Sonic_SENSOR):
            for payload in packet.getPayload():
                if(payload.getID() == MID_Ultra_Sonic_SENSOR):
                    buf = str(payload.getData())
                    data_Ultrasonic_Sensors = [int(float(buf[:3])),
                                            int(float(buf[3:6])),
                                            int(float(buf[6:9])),
                                            int(float(buf[9:12])),
                                            int(float(buf[12:15])),
                                            int(float(buf[15:]))]

while(True):
    Packet_send(Arduino_ID, REQUEST_Ultra_Sonic_SENSOR, MID_Ultra_Sonic_SENSOR)
    Packet_receive(Serial_Arduino)
    if(len(recv_parsing_packet) > 0):
        p = Received_packet()
        Ultrasonic_Data(p)
        print("Ultra_Sonic : " + str(data_Ultrasonic_Sensors))
        time.sleep(0.5)